# Previsão de Inadimplência em Cobranças

**Case Técnico — Processo Seletivo Programa de Aceleração em Ciência de Dados**
**Datarisk**

Julho/2026

---


## Objetivo

Este documento tem como objetivo apresentar o desenvolvimento de um modelo preditivo capaz de estimar a probabilidade de inadimplência de cobranças mensais realizadas a clientes, com base no histórico de comportamento de pagamento e nas características cadastrais e de perfil disponíveis.

Considera-se inadimplente todo pagamento realizado com **5 dias ou mais de atraso em relação à data de vencimento**. As previsões finais devem ser geradas para os registros da base `base_pagamentos_teste.csv`, contendo exclusivamente a probabilidade estimada de inadimplência (valores contínuos entre 0 e 1), sem classificação binária.


## Plano de Trabalho

O desenvolvimento da solução foi estruturado nas seguintes etapas:

**Etapa 1 - Análise Exploratória de Dados (EDA)**
Investigação individual de cada uma das quatro bases (cadastral, info, pagamentos_desenvolvimento e pagamentos_teste): estrutura, tipos, nulos, duplicatas, inconsistências lógicas e distribuições relevantes. Inclui também a construção e validação da variável target (inadimplência = atraso ≥ 5 dias), feita a partir da base de desenvolvimento.

**Etapa 2 - Merges e Feature Engineering**
União das quatro bases via `ID_CLIENTE` e `SAFRA_REF`, construindo a base consolidada de modelagem. Nesta etapa também são criadas novas variáveis, tratados componentes existentes (datas, categóricas, valores faltantes) e definidas estratégias para casos especiais (clientes sem histórico, inconsistências identificadas na EDA).

**Etapa 3 - Modelagem**
Treinamento e validação de modelo(s) de machine learning para estimar a probabilidade de inadimplência, com avaliação de métricas apropriadas ao problema.

**Etapa 4 - Interpretação dos Resultados**
Avaliação da performance do modelo, análise das variáveis mais relevantes e validação das previsões sob a ótica de negócio, seguida da geração do arquivo final `submissao_case.csv`.

Cada etapa é documentada a seguir, com as principais decisões técnicas e suas justificativas.

---


## Etapa 1 — Análise Exploratória de Dados

### 4.1. Base Cadastral (`base_cadastral.csv`)

Esta base reúne as informações cadastrais dos clientes, com granularidade de um registro por cliente (1.315 clientes únicos).

**O que foi feito:**
- Padronização de tipos e tratamento de valores nulos.
- Teste de hipóteses para verificar se a ausência de dados em algumas colunas tinha relação com o tipo de cliente (pessoa física vs jurídica).
- Verificação de duplicatas.

**Decisões tomadas:**
- A hipótese se confirmou para `SEGMENTO_INDUSTRIAL`: 100% dos nulos dessa coluna correspondem exatamente aos clientes PF, contra apenas 1,36% de nulos entre os PJ. Isso indica que a ausência não é um erro de preenchimento, mas sim estrutural (pessoa física não possui segmento industrial). Diante disso, os nulos de PF foram recodificados como uma categoria própria, `"PESSOA_FISICA"`, preservando essa distinção, enquanto os poucos nulos remanescentes em PJ foram tratados como `"Desconhecido"`.
- A mesma hipótese foi testada para `PORTE`, mas não se confirmou: a proporção de nulos é praticamente idêntica entre PF (3,03%) e PJ (3,12%), sugerindo que a ausência é aleatória e sem relação estrutural com o tipo de cliente. Nesse caso, todos os nulos foram tratados como `"Desconhecido"`.
- Para `DDD` e `DOMINIO_EMAIL`, sem hipótese estrutural aplicável, os nulos também foram tratados como `"Desconhecido"`.
- Não foram encontradas duplicatas de linha nem de `ID_CLIENTE`, confirmando que a granularidade da base está correta (um registro único por cliente).

### 4.2. Base de Informações Mensais (`base_info.csv`)

Esta base traz dados mensais de acompanhamento dos clientes, como renda do mês anterior e número de funcionários, com granularidade de um registro por cliente por safra (24.401 registros no total).

**O que foi feito:**
- Padronização de tipos e verificação da granularidade da base.
- Análise estatística das variáveis numéricas e investigação de valores potencialmente atípicos.

**Decisões tomadas:**
- Não foram encontradas duplicatas na combinação `ID_CLIENTE` + `SAFRA_REF`, confirmando a granularidade esperada da base.
- `RENDA_MES_ANTERIOR` apresenta 717 valores nulos (~3% da base) e uma distribuição assimétrica à direita (média de 288.751 bem acima da mediana de 240.998), o que é típico de variáveis de renda/faturamento. Essa assimetria foi registrada como um ponto de atenção para a etapa de feature engineering, onde uma transformação logarítmica poderá ser avaliada.
- Os 15 registros com `RENDA_MES_ANTERIOR` abaixo de 1.000 foram avaliados e considerados pouco expressivos (menos de 0,1% da base), sendo mantidos sem tratamento especial, apenas documentados como investigados.
- `NO_FUNCIONARIOS` apresenta 1.252 valores nulos (~5% da base) e uma distribuição mais simétrica (média de 117,8 próxima da mediana de 118).
- Os 214 registros com `NO_FUNCIONARIOS == 0` foram identificados como um ponto a ser melhor compreendido antes de qualquer tratamento definitivo, com um cruzamento iniciado contra a `FLAG_PF` da base cadastral para checar se a concentração ocorre entre clientes PF (o que validaria o valor como legítimo) ou entre PJ (o que sugeriria tratar como nulo disfarçado). **Esta investigação ficou pendente de conclusão** e será retomada antes da definição final do tratamento de nulos desta base.

### 4.3. Base de Pagamentos — Desenvolvimento (`base_pagamentos_desenvolvimento.csv`)

Esta base contém o histórico de cobranças já pagas, sendo a fonte para construção da variável target e para o desenvolvimento das features comportamentais do modelo.

**O que foi feito:**
- Cálculo do atraso entre pagamento e vencimento e construção da variável target (inadimplência = atraso ≥ 5 dias).
- Análise do balanceamento da target.
- Identificação de um pequeno grupo de registros com inconsistência lógica entre datas.
- Teste de hipótese relacionando essa inconsistência à ocorrência de inadimplência.

**Decisões tomadas:**
- Inicialmente, os 27 registros com inconsistência lógica de datas foram removidos da base, sob a hipótese de que se tratava de ruído de digitação. Contudo, ao investigar a distribuição da target apenas nesses registros, constatou-se que 26 dos 27 (96,3%) eram inadimplentes — uma proporção muito acima da média geral (7%). Diante desse achado, a decisão foi revertida: os registros foram reincorporados à base, e optou-se por criar, na etapa de feature engineering, uma variável binária (`FLAG_VENCIMENTO_INCONSISTENTE`) para capturar esse sinal, em vez de descartar informação potencialmente valiosa.
- Os nulos em `VALOR_A_PAGAR` não foram tratados nesta etapa. Observou-se que, entre os registros com valor ausente, a taxa de inadimplência é de 9,4% (contra 7% na base geral), sugerindo que a ausência do dado pode carregar sinal preditivo.

### 4.4. Base de Pagamentos — Teste (`base_pagamentos_teste.csv`)

Esta base contém as cobranças mais recentes, para as quais o modelo final deve gerar as probabilidades de inadimplência.

**O que foi feito:**
- Validação de estrutura e comparação com a base de desenvolvimento.
- Checagem de sobreposição temporal entre treino e teste.
- Verificação da mesma inconsistência lógica de datas e de nulos identificados na base de desenvolvimento.
- Análise da sobreposição de clientes entre as duas bases.

**Decisões tomadas:**
- Confirmada a ausência de sobreposição temporal entre treino e teste, validando a estrutura de validação temporal do problema (treinar no passado, prever o futuro).
- A presença do mesmo padrão de inconsistência de datas na base de teste reforçou a decisão de tratar esse padrão como uma feature (`FLAG_VENCIMENTO_INCONSISTENTE`) em vez de removê-lo, garantindo que a mesma lógica de tratamento seja aplicada de forma idêntica em treino e teste.
- A existência de 88 clientes sem histórico prévio (cold start) foi registrada como um ponto de atenção para a etapa de feature engineering, exigindo uma estratégia de fallback (por exemplo, imputação de features comportamentais com base em médias de segmento/perfil, já que não há histórico individual disponível para esses casos).

## Etapa 2: Engenharia de Features

Nesta etapa, o foco foi enriquecer a base de modelagem com variáveis derivadas capazes de capturar padrões comportamentais e temporais relevantes para a previsão de inadimplência, além de finalizar o tratamento de inconsistências identificadas na etapa de exploração. Um desafio adicional foi a descoberta de que alguns clientes na base teste não possuem cadastro, portanto novas features são essenciais para conseguirmos prever inadimplência nestes casos.

- **Validação de integridade dos merges**: antes de qualquer transformação, foi conferido que a junção das quatro bases (cadastral, info e pagamentos, tanto para desenvolvimento quanto para teste) preservou a contagem original de linhas (77.414 registros em desenvolvimento e 12.275 em teste), garantindo ausência de duplicações por chaves repetidas.

- **Análise dos nulos gerados pelos merges**: identificados novos nulos em `RENDA_MES_ANTERIOR` e `NO_FUNCIONARIOS`, decorrentes de combinações `ID_CLIENTE` + `SAFRA_REF` sem correspondência mensal na `base_info`. Também foi identificado um grupo de 21 clientes presentes apenas na base de teste, sem nenhum registro na base cadastral nem histórico prévio em desenvolvimento (cenário de cold start).

- **Estratégia de tratamento de nulos**:
  - `VALOR_A_PAGAR`, `RENDA_MES_ANTERIOR` e `NO_FUNCIONARIOS`: mantidos sem imputação, aproveitando a capacidade nativa de algoritmos baseados em árvore (LightGBM, XGBoost, CatBoost) de lidar com valores ausentes, evitando viés artificial.
  - Variáveis categóricas da cadastral (segmento, porte, domínio de e-mail, DDD, CEP) para os 21 clientes sem cadastro: preenchidas com a categoria "Desconhecido", consistente com o tratamento já aplicado na Etapa 1.

- **Criação de `NUMERO_COBRANCA_CLIENTE`**: variável ordinal que numera sequencialmente as cobranças de cada cliente ao longo do tempo, calculada a partir da concatenação de desenvolvimento e teste, garantindo que o histórico de desenvolvimento fosse corretamente considerado como passado dos clientes também presentes no teste. Uma flag de "primeira cobrança" foi cogitada e fica reservada como alternativa, caso a variável ordinal não demonstre poder discriminante suficiente na modelagem.

- **Features temporais**: extração de `DIA_COBRANCA` e `MES_COBRANCA` a partir de `DATA_VENCIMENTO`, com o objetivo de capturar possíveis padrões de sazonalidade na inadimplência (por exemplo, cobranças concentradas no início ou fim do mês, ou efeitos de determinados meses do ano).

- **Validação final**: todas as transformações foram checadas por meio de contagem de linhas, amostragem de registros e inspeção de casos extremos (como o cliente com maior número de cobranças), confirmando que as novas variáveis refletem padrões reais de comportamento, e não artefatos de processamento.